In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, io, warnings
from pathlib import Path
from scipy.signal import savgol_filter
sys.path.append(os.path.abspath('..'))
from src.wiserep import plot_spectra

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
target_name='SN2026jlm'
filedir=f'../output/{target_name}/spectrum/'
filepath=list(Path(filedir).glob('*.dat'))

plot_spectra(filepath, target_name, jupyter=True)

In [ ]:
import astrodash

# ── 抑制 astrodash / tensorflow / pandas 的冗长输出 ──
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示 TF 警告和错误
warnings.filterwarnings('ignore')           # 忽略 pandas/numpy 版本警告

# 抑制 astrodash 内部 print 的 20 行概率输出
import contextlib
class _SuppressPrint:
    def write(self, *a): pass
    def flush(self): pass
_orig_stdout = sys.stdout

filename_str = list(str(f) for f in filepath)

print(f'📂 光谱文件: {filename_str}')
print(f'✅ 文件存在: {Path(filename_str[0]).exists()}')

print("正在启动 DASH 进行模板匹配和深度学习分类...")

host_galaxy_z = 0.016738

try:
    # 临时吞掉 astrodash 内部 print 的概率数组
    sys.stdout = io.StringIO()
    classification = astrodash.Classify(
        filenames=filename_str,
        knownZ=host_galaxy_z,
    )
    # 获取最佳匹配 (返回 6 元组)
    ret = classification.list_best_matches()
    sys.stdout = _orig_stdout  # 恢复标准输出
    
    # ret[0]=bestMatchLists, ret[1]=redshifts, ret[2]=bestBroadTypes,
    # ret[3]=rlapLabels, ret[4]=matchesReliableLabels, ret[5]=redshiftErrs
    best_fits = ret[2]          # 最佳匹配列表
    redshifts = ret[1]          # 拟合红移数组
    
    if best_fits is not None and len(best_fits) > 0:
        best_match = best_fits[0]
        sn_name = best_match[0]
        sn_type = best_match[1]
        sn_age  = best_match[2]
        
        # 从 redshifts 数组取第一个谱的最佳红移 (shape 可能是 (n,) 或 (1, n))
        if redshifts is not None and len(redshifts) > 0:
            sn_z = float(redshifts[0]) if redshifts.ndim == 1 else float(redshifts[0, 0])
        else:
            sn_z = None
        
        print("-" * 40)
        print(f"🎯 DASH 最终分类结果 (基于已知红移 z={host_galaxy_z}):")
        print(f"超新星类型 (Type): {sn_type}")
        print(f"演化阶段 (Age): {sn_age} days (相对于光变极大期)")
        if sn_z is not None:
            print(f"拟合红移 (z): {sn_z:.4f}")
        else:
            print(f"拟合红移 (z): 未提供")
        print(f"最匹配的标准模板: {sn_name}")
        print("-" * 40)

        classification.plot_with_gui() 
    else:
        print("未找到合适的匹配模板。")

except Exception as e:
    sys.stdout = _orig_stdout  # 确保恢复
    import traceback
    traceback.print_exc()
    print(f"❌ DASH 运行过程中发生错误: {e}")

In [ ]:
# 假设前一步 DASH 告诉你这是一颗 Ia 型超新星 (有极强的 Si II 吸收线)
# 或者是一颗 II 型超新星 (有极强的 H-alpha 吸收线)

# 1. 选择我们要测量的特征谱线 (静止波长)
# 如果是 Ia 型，取消注释这行：
# rest_wave = 6355.0  # 硅离子 Si II 的特征波长
# 如果是 II 型，取消注释这行：
rest_wave = 6562.8  # 氢的巴耳末系 H-alpha 波长

c = 299792.458 # 光速 (km/s)

# 2. 去除红移 (De-redshift)
# 利用 Cell 2 中拟合出来的红移 sn_z，把观测波长还原到超新星的静止参考系
wave_rest_frame = wave / (1 + sn_z)

# 3. 利用 NumPy 切片寻找吸收谷 (Absorption Trough)
# 膨胀速度通常在 5000 ~ 20000 km/s，对应的波长蓝移大约在 100 ~ 400 埃
search_min = rest_wave - 500 # 搜索下限
search_max = rest_wave       # 搜索上限

# 使用布尔索引 (NumPy的精髓) 截取特征谱线附近的数据
mask = (wave_rest_frame > search_min) & (wave_rest_frame < search_max)
local_wave = wave_rest_frame[mask]
local_flux = flux_smoothed[mask]

# 找到这段区间内流量最小的点 (即吸收线中心)
min_idx = np.argmin(local_flux)
obs_abs_wave = local_wave[min_idx]

# 4. 计算膨胀速度 (非相对论多普勒公式 v = c * Δλ / λ)
v_exp = c * (rest_wave - obs_abs_wave) / rest_wave

print(f"谱线静止波长: {rest_wave} A")
print(f"实际观测到的吸收谷波长: {obs_abs_wave:.1f} A")
print(f"🚀 结论：抛射物膨胀速度 v_exp = {v_exp:.0f} km/s")

# 5. 画个局部放大图展示成果
plt.figure(figsize=(8, 4))
plt.plot(local_wave, local_flux, 'b-', label='Smoothed Spectrum')
plt.axvline(rest_wave, color='red', linestyle='--', label='Rest Wavelength (Zero Velocity)')
plt.axvline(obs_abs_wave, color='green', linestyle=':', label='Observed Minimum')
plt.title(f'P-Cygni Absorption Feature (v = {v_exp:.0f} km/s)')
plt.xlabel('Rest-frame Wavelength ($\AA$)')
plt.legend()
plt.show()